# Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile

zip_path = '/content/drive/MyDrive/bratsmri.zip'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.printdir()  # prints all files inside the ZIP


# Extract Features from the FLAIR MODALITY

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
import nibabel as nib
import cv2
from skimage.feature import local_binary_pattern
from skimage import measure


zip_path = '/content/drive/MyDrive/bratsmri.zip'
extract_path = '/content/brats_dataset'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

base_path = os.path.join(extract_path, 'MICCAI_BraTS_2019_Data_Training')


radius = 1
n_points = 8 * radius

def extract_features(img2d):
    img = cv2.resize(img2d, (128, 128))

    features = []

    # Intensity features
    features.extend([np.mean(img), np.std(img), np.max(img), np.min(img), np.median(img)])

    # LBP features
    lbp = local_binary_pattern(img, n_points, radius, method='uniform')
    n_bins = int(lbp.max() + 1)
    hist, _ = np.histogram(lbp, bins=n_bins, range=(0, n_bins), density=True)
    features.extend(hist.tolist())

    # Shape features
    _, mask = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    labels = measure.label(mask)
    props = measure.regionprops(labels)
    if props:
        largest = max(props, key=lambda x: x.area)
        features.extend([largest.area, largest.perimeter, largest.eccentricity, largest.solidity])
    else:
        features.extend([0,0,0,0])

    return features


def process_brats_folder(folder_path, label):
    X, y = [], []
    for patient in os.listdir(folder_path):
        patient_path = os.path.join(folder_path, patient)
        if not os.path.isdir(patient_path):
            continue

        flair_file = [f for f in os.listdir(patient_path) if f.endswith("_flair.nii")]
        if not flair_file:
            continue

        img_path = os.path.join(patient_path, flair_file[0])
        nii = nib.load(img_path)
        data = nii.get_fdata()

        # Pick middle slice (you can also loop over all slices)
        mid_slice = data.shape[2] // 2
        slice_img = data[:, :, mid_slice]

        # Normalize to 0-255
        slice_img = cv2.normalize(slice_img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

        feats = extract_features(slice_img)
        X.append(feats)
        y.append(label)

    return X, y

# HGG = 1, LGG = 0
X_hgg, y_hgg = process_brats_folder(os.path.join(base_path, 'HGG'), label=1)
X_lgg, y_lgg = process_brats_folder(os.path.join(base_path, 'LGG'), label=0)

X = X_hgg + X_lgg
y = y_hgg + y_lgg

# -----------------------------
# Save CSV
# -----------------------------
feature_names = ['mean','std','max','min','median']
lbp_cols = [f'lbp_{i}' for i in range(int(n_points+2))]
shape_cols = ['area','perimeter','eccentricity','solidity']

df = pd.DataFrame(X, columns=feature_names + lbp_cols + shape_cols)
df['label'] = y

csv_path = '/content/drive/MyDrive/BraTS_features.csv'
df.to_csv(csv_path, index=False)

print("Features saved to:", csv_path)
print("Rows:", df.shape[0])
